<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


<h2>实验：使用 CNN 进行图像分类


预计所需时间：**60** 分钟


## 概述


在本实验中，你将使用<a href="https://cs231n.github.io/transfer-learning/">迁移学习</a>训练一个用于图像分类的深度神经网络，并尝试不同的超参数。


## 目标


在本实验中，你将训练一个最先进的图像分类器。在实践中，很少有人从头开始训练整个卷积神经网络（使用随机初始化），因为拥有足够大的数据集相对罕见，而且从头训练需要大量计算资源。
相反，常见的做法是使用已经在非常大的数据集（如 ResNet）上预训练过的卷积网络，然后在你自己的数据集上进行微调。我们将使用预训练的卷积网络作为特征提取器，只训练输出层。
一般来说，100-200 张图像就能给你一个不错的起点，只需要大约半个小时。通常，添加的图像越多，结果越好，但耗时越长，改进速度也会下降。


# 目录


本笔记本分为以下几个部分：
-   [安装与导入库](#Install-and-Import-Libraries)
-   [图像处理与数据加载以准备数据集](#Image-Processing-and-Load-Data-for-Dataset-preparation)
-   [加载模型并训练](#Load-Model-and-Train)
-   [练习](#Practice-Exercise)


* * *


## 安装与导入库


**安装可能需要一些时间，请耐心等待。**


In [ ]:
# 核心库
!pip install numpy pandas matplotlib tqdm pillow --quiet


In [ ]:
# 进度条和交互组件
!pip install ipywidgets --quiet
# 如果你使用 JupyterLab，也请启用 widget 扩展
#!jupyter nbextension enable --py widgetsnbextension


In [ ]:
# PyTorch（CPU 版本）
!pip install torch torchvision --quiet


**导入库并定义辅助函数**


用于操作系统和云的库


In [ ]:
import os
import uuid
import shutil
import json
import copy
from datetime import datetime
import zipfile
import io
import requests
import random


用于数据处理和可视化的库


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from matplotlib.pyplot import imshow
from tqdm import tqdm
from ipywidgets import IntProgress
import time 


深度学习库


In [ ]:
import torch
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader,random_split
from torch.optim import lr_scheduler
from torchvision import transforms
import torch.nn as nn
torch.manual_seed(0)
from torchvision.datasets import ImageFolder


**绘制训练代价和验证准确率：**

`plot_stuff` 函数用于可视化模型的训练表现，通过在同一图中使用两个不同的 y 轴绘制训练损失和验证准确率。

左侧 y 轴上用红色显示的训练损失表示模型在训练过程中最小化误差的能力。

右侧 y 轴上用蓝色显示的验证准确率反映模型在未见数据上的表现。

通过将这两个指标一起绘制，该函数有助于监控学习过程、识别趋势并检测过拟合等问题——即模型在训练数据上表现良好，但在验证数据上表现不佳。
这种可视化对于理解训练策略的有效性以及明智地调整模型架构或超参数非常有价值。


In [ ]:
def plot_stuff(COST, ACC):
    """
    使用两个 y 轴在同一图中绘制训练代价（损失）和验证准确率。
    
    参数：
    COST（列表或数组）：每次迭代（或轮次）的总训练损失
    ACC（列表或数组）：每次迭代（或轮次）的验证准确率
    """
    
    # 创建新图和主轴 (ax1)
    fig, ax1 = plt.subplots()
    
    # 在左侧主轴 y 轴上绘制训练损失
    color = 'tab:red'
    ax1.plot(COST, color=color)
    ax1.set_xlabel('迭代', color=color)            # x 轴标签
    ax1.set_ylabel('总损失', color=color)           # 左侧 y 轴标签
    ax1.tick_params(axis='y', labelcolor=color)         # 设置 y 轴刻度颜色

    # 创建共享同一 x 轴的次 y 轴 (ax2)
    ax2 = ax1.twinx()
    
    # 在右侧次 y 轴上绘制验证准确率
    color = 'tab:blue'
    ax2.set_ylabel('准确率', color=color)             # 右侧 y 轴标签
    ax2.plot(ACC, color=color)
    ax2.tick_params(axis='y', labelcolor=color)

    # 调整布局以防止 y 轴标签被裁剪
    fig.tight_layout()
    
    # 显示组合图
    plt.show()


**绘制变换后的图像：**

在预处理过程中使用 `transforms.Normalize` 时（例如对于像 ResNet 这样的预训练 CNN），图像会使用均值和标准差进行归一化。但如果直接显示，图像看起来会很奇怪。

该函数会撤销这种归一化，使图像显示为正常的 RGB 图像以便可视化，特别适用于：

检查图像处理流程。

显示数据集中的样本。

调试预测或模型输出。


In [ ]:
def imshow_(inp, title=None):
    """
    显示撤销归一化后的张量图像。
    
    参数：
    - inp（Tensor）：形状为 [C, H, W] 的图像张量，通常已归一化。
    - title（字符串，可选）：图像显示的标题。
    """
    # 从 [C, H, W] 转换为 [H, W, C] 并转为 NumPy 数组
    inp = inp.permute(1, 2, 0).numpy()
    print("图像形状:", inp.shape)

    # 撤销归一化（ImageNet 均值和标准差）
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean

    # 将值裁剪到 [0, 1] 范围以便显示
    inp = np.clip(inp, 0, 1)

    # 显示图像
    plt.imshow(inp)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)  # 短暂暂停以更新 GUI
    plt.show()


**如果 CUDA 可用，将设备定义为第一个可见的 CUDA 设备：**


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("设备类型为", device)


## 图像处理与数据加载以准备数据集


在本节中，我们将通过调整图像大小、将其转换为张量以及对图像通道进行归一化来预处理数据集。这些是图像数据的标准预处理步骤。此外，我们将对训练数据集应用数据增强以提高泛化能力。测试（或验证）数据集的预处理步骤相同，只是不应用数据增强，因为我们希望在未经修改的图像上评估模型。


**用于归一化的均值和标准差值（基于 ImageNet）**

```python
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# 为训练数据集定义组合变换
composed = transforms.Compose([
    transforms.Resize((224, 224)),              # 调整大小以匹配模型输入
    transforms.RandomHorizontalFlip(),          # 随机水平翻转图像
    transforms.RandomRotation(degrees=5),       # 小范围随机旋转
    transforms.ToTensor(),                      # 将图像转换为张量 [C, H, W]
    transforms.Normalize(mean, std)             # 使用预定义的均值和标准差进行归一化
])
```


**加载数据以准备数据集**


下载数据：


In [ ]:
# ZIP 文件 URL
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/QkkP2jRxVvxHKMPg8lnkwQ/transfer-learning-with-cnn-15-2025-06-24-t-12-35-01-829-z.zip"

# 向 URL 发送 GET 请求
response = requests.get(url)

# 检查请求是否成功
if response.status_code == 200:
    # 从下载内容中打开 zip 文件
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        zip_ref.extractall("hotdog_nothotdog")  # 解压到目标文件夹
    print("下载并解压完成。")
else:
    print("下载文件失败：", response.status_code)


将图像数据划分为训练集和验证集


设置路径


In [ ]:
# 设置路径
source_dir = "hotdog_nothotdog/transfer-learning-with-cnn-15-2025-06-24-t-12-35-01-829-z"  # 包含图像和标注文件的文件夹
annotations_file = os.path.join(source_dir, "_annotations.json")  # 如有需要请更新名称


加载标注


In [ ]:
# 加载标注
with open(annotations_file, "r") as f:
    annotations = json.load(f)


设置参数


In [ ]:
# 参数
train_ratio = 0.9
output_dir = "dataset"  # 最终输出根目录


准备标签到图像的映射


In [ ]:
# 准备标签 -> 图像列表
label_to_images = {}

for filename, entry in annotations["annotations"].items():
    label = entry[0]["label"]
    label_to_images.setdefault(label, []).append(filename)


打乱并划分为训练/验证集

**划分数据集：90% 训练，10% 验证**
我们定义 `train_ratio = 0.9` 并将其应用于每个类别。打乱后的前 90% 图像用于训练，剩余 10% 用于验证。


In [ ]:
# 将每个类别的图像打乱并划分为训练集和验证集
for label, image_list in label_to_images.items():
    random.shuffle(image_list)  # 打乱图像列表以随机化划分
    
    # 计算训练图像数量（例如总数的 90%）
    train_cutoff = int(len(image_list) * train_ratio)
    
    # 将图像列表划分为训练集和验证集
    train_images = image_list[:train_cutoff]
    val_images = image_list[train_cutoff:]

    # 遍历两个划分：'train' 和 'val'
    for split, split_images in zip(["train", "val"], [train_images, val_images]):
        
        # 为当前划分和标签创建输出目录
        # 例如：dataset/train/hotdog 或 dataset/val/nothotdog
        out_path = os.path.join(output_dir, split, label)
        os.makedirs(out_path, exist_ok=True)  # 如果目录不存在则创建

        # 将每张图像从源目录复制到相应的划分文件夹
        for img_name in split_images:
            src = os.path.join(source_dir, img_name)  # 源图像的完整路径
            dst = os.path.join(out_path, img_name)    # 目标路径
            shutil.copy2(src, dst)  # 复制图像（保留元数据）

# 所有图像复制完成后输出完成信息
print("训练/验证划分完成。")


应用变换


In [ ]:
# 定义要应用于每张图像的一系列变换
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 将所有图像调整为 224×224 像素（预训练模型的标准输入大小）
    transforms.ToTensor(),          # 将 PIL 图像转换为形状为 [C, H, W]、取值在 [0, 1] 的 PyTorch 张量
    transforms.Normalize(           # 使用 ImageNet 的均值和标准差对每个通道（RGB）进行归一化
        [0.485, 0.456, 0.406],      # 红、绿、蓝通道的均值
        [0.229, 0.224, 0.225]       # 红、绿、蓝通道的标准差
    )
])

# 从文件夹结构加载训练数据集并应用定义的变换
train_dataset = ImageFolder("dataset/train", transform=transform)

# 从文件夹结构加载验证数据集并应用相同的变换
#（注意：这里不应用数据增强——只进行大小调整、张量转换和归一化）
val_dataset = ImageFolder("dataset/val", transform=transform)


我们可以从验证集中绘制部分数据集：


In [ ]:
i = 0
for x, y in val_dataset:                     # 遍历验证数据集
    imshow_(x, f"y = {y}")               # 显示图像及其标签
    i += 1                               # 计数器递增
    if i == 3:                           # 显示 3 张图像后停止
        break


### 超参数


尝试不同的超参数：


<b>Epoch（轮次）</b>表示遍历整个训练数据集的次数，这里我们将轮次数设置为 10：


In [ ]:
n_epochs=10


<b>Batch size（批量大小）</b>是一次迭代中使用的训练样本数量。如果批量大小等于训练集中的样本总数，那么每个轮次只有一次迭代。在随机梯度下降中，批量大小设置为 1。32-512 个数据点的批量大小似乎是一个不错的值，更多信息请查看以下<a href="https://arxiv.org/abs/1609.04836?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-cvstudio-2021-01-01">链接</a>。


In [ ]:
batch_size=32


 <b>Learning rate（学习率）</b>用于神经网络的训练。学习率是一个小的正值超参数，通常在 0.0 到 1.0 之间。


In [ ]:
lr=0.000001


<b>Momentum（动量）</b>是一种与梯度下降配合使用的优化技术，用于加速收敛（当模型训练过程稳定并停止显著改进时）、减少陡峭山谷中的震荡，并帮助逃离局部最小值。


In [ ]:
momentum=0.9


如果你设置为 <code>lr_scheduler=True</code>，则每个轮次都会使用学习率调度器，将学习率范围从最大值或最小值进行变化。学习率通常会随时间衰减。


In [ ]:
lr_scheduler=True
base_lr=0.001
max_lr=0.01


# 加载模型并训练


`train_model()` 函数将训练模型：

`train_model()` 执行的步骤

- 初始化跟踪变量
  
         设置列表以存储每个轮次的训练损失和验证准确率。
         将初始模型权重保存为目前为止的最佳模型。
  
- 遍历轮次

         对于每个轮次（遍历整个训练数据集一次）：

**训练阶段：**
- 遍历 train_loader 中的批次

         将输入数据 (x) 和标签 (y) 移动到适当的设备（CPU 或 GPU）。
         将模型设置为训练模式。
         通过模型执行前向传播。
         使用损失函数计算损失。
         调用 backward() 计算梯度。
         调用 optimizer.step() 更新权重。
         使用 optimizer.zero_grad() 重置梯度。

- 计算该轮次的平均训练损失

         将批次损失的均值追加到 loss_list。

- 更新学习率（如果提供了调度器）

         调用 scheduler.step() 调整学习率。

**验证阶段：**
- 在验证集上评估模型

         将模型设置为评估模式。
         使用 torch.no_grad() 禁用梯度计算。
         预测标签并与真实标签比较。
         累积正确预测总数以计算准确率。

- 跟踪并保存最佳模型

         如果当前验证准确率优于之前的最佳值，则保存模型权重。

- 输出轮次指标（可选）

         如果 print_ 为 True，则输出当前学习率、验证损失和准确率。

- 加载最佳模型权重

         所有轮次结束后，加载表现最佳的模型权重（基于验证准确率）。

- 返回结果

         返回验证准确率列表、训练损失列表和训练好的模型。


In [ ]:
def train_model(model, train_loader, validation_loader, criterion, optimizer, n_epochs, print_=True):
    loss_list = []        # 存储每个轮次的平均训练损失
    accuracy_list = []    # 存储每个轮次的验证准确率
    correct = 0

    n_test = len(val_dataset)  # 验证样本总数
    accuracy_best = 0      # 跟踪最佳验证准确率
    best_model_wts = copy.deepcopy(model.state_dict())  # 备份最佳模型权重

    print("第一个轮次可能需要几分钟")

    for epoch in tqdm(range(n_epochs)):  # 遍历每个轮次
        loss_sublist = []  # 存储该轮次的每个批次损失

        # 训练阶段
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            model.train()  # 将模型设置为训练模式

            z = model(x)   # 前向传播
            loss = criterion(z, y)  # 计算损失
            loss_sublist.append(loss.item())

            loss.backward()       # 反向传播
            optimizer.step()      # 更新权重
            optimizer.zero_grad() # 重置梯度

        print(f"轮次 {epoch + 1} 完成")

        # 如果定义了学习率调度器，则调整学习率
        scheduler.step()

        # 存储该轮次的平均训练损失
        loss_list.append(np.mean(loss_sublist))

        # 验证阶段
        correct = 0
        model.eval()  # 将模型设置为评估模式
        with torch.no_grad():
            for x_test, y_test in validation_loader:
                x_test, y_test = x_test.to(device), y_test.to(device)
                z = model(x_test)
                _, yhat = torch.max(z.data, 1)
                correct += (yhat == y_test).sum().item()

        accuracy = correct / n_test
        accuracy_list.append(accuracy)

        # 保存最佳模型
        if accuracy > accuracy_best:
            accuracy_best = accuracy
            best_model_wts = copy.deepcopy(model.state_dict())

        # 输出训练进度
        if print_:
            print("学习率:", optimizer.param_groups[0]['lr'])
            print(f"验证损失（轮次 {epoch + 1}）: {np.mean(loss_sublist):.4f}")
            print(f"验证准确率（轮次 {epoch + 1}）: {accuracy:.4f}")

    # 返回前加载最佳模型权重
    model.load_state_dict(best_model_wts)
    return accuracy_list, loss_list, model


 加载预训练模型 resnet18。将 pretrained 参数设置为 true。


In [ ]:
model = models.resnet18(pretrained=True)


我们只训练网络的最后一层，将参数 <code>requires_grad</code> 设置为 <code>False</code>，这样网络就成为固定的特征提取器。


In [ ]:
for param in model.parameters():
        param.requires_grad = False
    


类别数量


In [ ]:
n_classes = len(train_dataset.classes)
print(n_classes)


将神经网络的输出层 model.fc 替换为 nn.Linear 对象，以对 <code>n_classes</code> 个不同类别进行分类。对于 in_features 参数，请记住最后一个隐藏层有 512 个神经元。


In [ ]:
model.fc = nn.Linear(512, n_classes)


设置设备类型


In [ ]:
model.to(device)


交叉熵损失（或对数损失）衡量分类模型的性能，在一个对象类别中结合了 LogSoftmax。它适用于训练具有 C 个类别的分类问题。


In [ ]:
criterion = nn.CrossEntropyLoss()


创建训练加载器和验证加载器对象。


In [ ]:
train_loader = torch.utils.data.DataLoader(dataset=train_dataset , batch_size=batch_size,shuffle=True)
validation_loader= torch.utils.data.DataLoader(dataset=val_dataset , batch_size=1)


使用 optim 包定义一个优化器，它将为我们更新模型的权重。


In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)


我们使用<a href='https://arxiv.org/pdf/1506.01186.pdf'>循环学习率</a>


In [ ]:
if lr_scheduler:
    scheduler = torch.optim.lr_scheduler.CyclicLR(
        optimizer,
        base_lr=0.001,        # 最小学习率
        max_lr=0.01,          # 最大学习率
        step_size_up=5,       # 学习率从 base_lr 增加到 max_lr 所需的步数
        mode="triangular2"    # 学习率按三角周期变化，每个周期后将 max_lr 减半
    )


现在我们要训练模型，对于给定的图像，这需要大约 25 分钟，具体时间取决于你的数据集


In [ ]:
# 开始计时
start_datetime = datetime.now()
start_time = time.time()

# 训练模型
accuracy_list, loss_list, model = train_model(
    model, train_loader, validation_loader, criterion, optimizer, n_epochs=n_epochs
)

# 结束计时
end_datetime = datetime.now()
elapsed_time = time.time() - start_time

# 输出结果
print("训练完成。")
print(f"开始时间     : {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"结束时间       : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"已用时间   : {elapsed_time:.2f} 秒")


将模型保存到 model.pt


In [ ]:
# 将模型保存到 model.pt
torch.save(model.state_dict(), 'model.pt')


绘制训练代价和验证准确率，你可以通过获取更多数据来改进结果。


In [ ]:
plot_stuff(loss_list,accuracy_list)


# 练习

### 使用上传的图像测试模型


上传你的图像，看看是否能被正确分类。
<p><b>如何上传图像：</b></p>
使用上传按钮从本地机器上传图像：
<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-SkillsNetwork/images/instruction.png" width="300"  />
</center>


图像现在会出现在你当前工作的目录中。要在新单元格中读取图像，请使用 <code>cv2.imread</code> 并指定其名称。例如，我将 <code>anothercar.jpg</code> 上传到当前工作目录——<code>cv2.imread("anothercar.jpg")</code>。

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-SkillsNetwork/images/instruction2.png" width="300"  />
</center>


或者使用下面的图像进行测试。


In [ ]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/eTwtaXXHQQkxDlkFWHqRNw/test.jpg"
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/xGq5q1hvl7HQLhmVGAOfzQ/test1.jpg"


定义类别名称并加载模型


In [ ]:
# 定义类别名称（与训练时一致）
class_names = ['hotdog', 'nothotdog']
# 创建与训练时相同的模型架构
model = models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(model.fc.in_features, 2)  # 2 个类别：hotdog / nothotdog

# 加载训练好的权重
model.load_state_dict(torch.load("model.pt", map_location=torch.device('cpu')))
model.eval()  # 设置为评估模式


定义图像变换


In [ ]:
# 定义图像变换（必须与训练时一致）
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 调整图像大小
    transforms.ToTensor(),  # 转换为张量
    transforms.Normalize([0.485, 0.456, 0.406],  # 归一化（与 ImageNet/预训练模型相同）
                         [0.229, 0.224, 0.225])
])


加载并预处理图像

将下面的 your_uploaded_file 替换为你目录中显示的图像名称。如果你使用的是笔记本中提供的下载图像，则使用 `test.jpg` 或 `test1.jpg`


In [ ]:
image_path = "test.jpg"  # 替换为你的图像路径

# 打开并转换为 RGB
image = Image.open(image_path).convert("RGB")

# 应用变换
input_tensor = transform(image).unsqueeze(0)  # 添加批次维度


进行预测并显示结果


In [ ]:
with torch.no_grad():
    outputs = model(input_tensor)
    predicted_class = torch.argmax(outputs, 1).item()
# 显示结果
print(f"The image was classified as: {class_names[predicted_class]}")
# 显示带有预测标签的图像
plt.imshow(image)  # 原始 PIL 图像
plt.title(f"Predicted: {class_names[predicted_class]}")
plt.axis("off")
plt.show()


### 恭喜！你已完成使用迁移学习的图像分类实验。
你成功地利用预训练的深度神经网络构建了一个有效的图像分类器，尝试了超参数，并在应用迁移学习方面获得了实践经验——迁移学习是现代计算机视觉中广泛使用的方法。


## 作者


Joseph Santarcangelo

[Sathya Priya](https://www.linkedin.com/in/sathya-priya-06120a17a/) 



<!--## 变更日志-->


<!--| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2025-06-25        | 0.4    | Sathya Priya| Created and Converted the lab to JupyterCurrent notebook |
| 2021-05-25        | 0.3     | Yasmine    | Modifies Multiple Areas |
| 2021-05-25        | 0.3     | Kathy      | Modified Multple Areas. |
| 2021-03-08        | 0.2     | Joseph     | Modified Multiple Areas |
| 2021-02-01        | 0.1     | Joseph     | Modified Multiple Areas |-->


<h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
